In [ ]:
import numpy as np
import pandas as pd

In [ ]:
import os, shutil

In [ ]:
train_path='../image-dataset/dogs-vs-cats/train'

In [ ]:
animal=[]
img_path=[]
for file in os.listdir(train_path):
    animal.append(str(file.split('.')[0]))
    img_path.append(file)

In [ ]:
df=pd.DataFrame({'animal': animal,'img':img_path})

In [ ]:
df.head()

,animal,img
0,cat,cat.0.jpg
1,cat,cat.1.jpg
2,cat,cat.10.jpg
3,cat,cat.100.jpg
4,cat,cat.1000.jpg


In [ ]:
def convert(value):
    if value=='cat':
        return 1
    else:
        return 0
df['animal']=df['animal'].apply(convert)

In [ ]:
df.head()

,animal,img
0,1,cat.0.jpg
1,1,cat.1.jpg
2,1,cat.10.jpg
3,1,cat.100.jpg
4,1,cat.1000.jpg


In [ ]:
train_df = df.sample(frac=1,random_state=0).iloc[:20000]
test_df = df.sample(frac=1,random_state=0).iloc[20000:]

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
train_datagen=ImageDataGenerator(rescale=1./255,
                                 rotation_range=30,
                                 width_shift_range=0.2,
                                 height_shift_range=0.2,
                                 shear_range=0.2,
                                 zoom_range=0.2,
                                 horizontal_flip=True)

test_datagen=ImageDataGenerator(rescale=1./255)

In [ ]:
train_generator = train_datagen.flow_from_dataframe(train_df,
                                                    directory=train_path,
                                                    x_col='img',
                                                    y_col='animal',
                                                    target_size=(128,128),
                                                    class_mode='raw')

test_generator = test_datagen.flow_from_dataframe(test_df,
                                                    directory=train_path,
                                                    x_col='img',
                                                    y_col='animal',
                                                    target_size=(128,128),
                                                  class_mode='raw')

Found 20000 validated image filenames.
Found 5000 validated image filenames.


## feature extraction data aug

In [ ]:
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense,Flatten
from keras.applications.vgg16 import VGG16

In [ ]:
conv_base=VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(128,128,3)
)

In [ ]:
conv_base.trainable = True

set_trainable = False

for layer in conv_base.layers:
  if layer.name == 'block5_conv1':
    set_trainable = True
  if set_trainable:
    layer.trainable = True
  else:
    layer.trainable = False

for layer in conv_base.layers:
  print(layer.name,layer.trainable)

In [ ]:
model = Sequential()

model.add(conv_base)
model.add(Flatten())
model.add(Dense(256,activation='relu'))
model.add(Dense(1,activation='sigmoid'))

In [ ]:
model.compile(
    optimizer=keras.optimizers.RMSprop(lr=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
  )

In [ ]:
history = model.fit(train_ds,epochs=10,validation_data=validation_ds)